In [ ]:
import os
import warnings
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
DATA_PATH       = ""   # e.g. "data/combined.csv"
METRICS_DIR     = ""   # e.g. "reports/metrics/ARIMAX"
RESULTS_DIR     = ""   # e.g. "reports/results/ARIMAX"
FIGURES_DIR     = ""   # e.g. "reports/figures/ARIMAX"
MODEL_DIR       = ""   # e.g. "data/models"
OOF_DIR         = ""   # e.g. "data/oof"
SUMMARY_METRICS = ""   # e.g. "reports/metrics/ARIMAX/all_metrics.csv"

SYMBOLS   = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
EXOG_COLS = ["dxy", "fed_rate", "gold", "oil", "sp500"]  # macro exogenous variables

TRAIN_RATIO = 0.8
N_FOLDS     = 5
GAP_DAYS    = 30   # purge gap between train end and val start to avoid leakage

In [ ]:
def split_data(df: pd.DataFrame):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    split_idx = int(len(df) * TRAIN_RATIO)
    train_df  = df.iloc[:split_idx].reset_index(drop=True)
    test_df   = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df

In [ ]:
def find_best_order(endog: pd.Series, exog: pd.DataFrame) -> tuple:
    log.info("    Running auto_arima to find best (p,d,q)...")
    model = auto_arima(
        endog,
        exogenous=exog,
        seasonal=False,          # no seasonal component — stock prices have no clear seasonality
        test="adf",              # use adf test to auto-select differencing order d
        d=None,                  # let auto_arima determine d to handle non-stationarity
        stepwise=True,
        information_criterion="aic",
        error_action="ignore",
        suppress_warnings=True,
        trace=False,
        max_p=3, max_q=3,        # cap order to avoid overfitting
    )
    order = model.order          # (p, d, q) — d handles non-stationarity via differencing
    log.info(f"    Best order={order}")
    return order

In [ ]:
def run_oof(train_df: pd.DataFrame, order: tuple, ticker: str) -> pd.DataFrame:
    n         = len(train_df)
    fold_size = n // (N_FOLDS + 1)
    oof_preds = np.full(n, np.nan)  # nan everywhere, only val windows will be filled

    log.info(f"    OOF: {N_FOLDS} folds, fold_size={fold_size}, gap={GAP_DAYS}")

    for k in range(1, N_FOLDS + 1):
        train_end = k * fold_size
        val_start = train_end + GAP_DAYS   # skip gap days to prevent leakage through rolling features
        val_end   = val_start + fold_size

        if val_end > n:
            break

        fold_train = train_df.iloc[:train_end]
        fold_val   = train_df.iloc[val_start:val_end]

        try:
            result = ARIMA(
                fold_train["Close"],
                exog=fold_train[EXOG_COLS],   # macro variables as exogenous regressors
                order=order,
            ).fit()

            forecast = result.forecast(
                steps=len(fold_val),
                exog=fold_val[EXOG_COLS],     # must pass future exog values when forecasting
            )
            oof_preds[val_start:val_end] = forecast.values

            mae = np.mean(np.abs(forecast.values - fold_val["Close"].values))
            log.info(f"      Fold {k}: train=[0:{train_end}]  "
                     f"val=[{val_start}:{val_end}]  MAE={mae:.4f}")

        except Exception as e:
            log.warning(f"      Fold {k} failed: {e}")
            oof_preds[val_start:val_end] = np.nan

    return pd.DataFrame({
        "Ticker":                 ticker,
        "Date":                   train_df["Date"].values,
        "Close":                  train_df["Close"].values,
        "ARIMAX_Predicted_Close": oof_preds,   # nan at gap regions between folds
    })

In [ ]:
def predict_test(final_result, test_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    forecast = final_result.forecast(
        steps=len(test_df),
        exog=test_df[EXOG_COLS],
    )
    return pd.DataFrame({
        "Ticker":                 ticker,
        "Date":                   test_df["Date"].values,
        "Close":                  test_df["Close"].values,
        "ARIMAX_Predicted_Close": forecast.values,
    })

In [ ]:
def calc_metrics(
    df: pd.DataFrame,
    actual_col: str = "Close",
    pred_col:   str = "ARIMAX_Predicted_Close",
    ticker_col: str = "Ticker",
) -> pd.DataFrame:

    # recurse per ticker when df contains multiple tickers
    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return (
            df.groupby(ticker_col)
              .apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None))
              .reset_index()
        )

    df = df.copy()
    df = df.dropna(subset=[actual_col, pred_col])   # drop nan rows from oof gap regions
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)  # +1 up, -1 down, 0 flat
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)                     # directional accuracy

    mask = actual_dir != 0                                            # exclude flat days from tpa
    tpa  = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    # volatility rmse: compare de-meaned series so level bias doesn't inflate the error
    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE":    mse,
        "MAE":    mae,
        "MAPE":   mape,
        "RMSE":   rmse,
        "R2":     r2,
        "DA":     da,
        "TPA":    tpa,
        "V-RMSE": v_rmse,
    }])

In [ ]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker  = df["Ticker"].iloc[0]
    df_plot = df.dropna(subset=["ARIMAX_Predicted_Close"])

    os.makedirs(save_dir, exist_ok=True)
    sns.set(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # top panel: actual vs predicted close price
    axes[0].plot(df_plot["Date"], df_plot["Close"],
                 label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(df_plot["Date"], df_plot["ARIMAX_Predicted_Close"],
                 label="ARIMAX Predicted", color="#d62728", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"ARIMAX predicted results: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis="x", rotation=30)

    # bottom panel: residual error per day
    error  = df_plot["Close"] - df_plot["ARIMAX_Predicted_Close"]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")   # green = under-predicted, red = over-predicted
    axes[1].bar(df_plot["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    log.info(f"    Plot saved: {save_path}")

In [ ]:
def train_one_ticker(ticker: str) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    train_df, test_df = split_data(df)
    log.info(f"  Train: {len(train_df)} rows  Test: {len(test_df)} rows")

    order = find_best_order(train_df["Close"], train_df[EXOG_COLS])

    oof_df = run_oof(train_df, order, ticker)

    # refit on full train set to use all available signal before predicting test
    log.info("    Refitting on full train set...")
    final_result = ARIMA(
        train_df["Close"],
        exog=train_df[EXOG_COLS],
        order=order,
    ).fit()

    test_result_df = predict_test(final_result, test_df, ticker)

    # oof metrics are unbiased — model never saw val rows during training
    metrics_df = calc_metrics(oof_df)
    metrics_df.insert(0, "Ticker", ticker)
    log.info(f"  OOF  — MAE={metrics_df['MAE'].iloc[0]:.4f}  "
             f"DA={metrics_df['DA'].iloc[0]:.2%}  "
             f"R2={metrics_df['R2'].iloc[0]:.4f}")

    test_metrics = calc_metrics(test_result_df)
    log.info(f"  Test — MAE={test_metrics['MAE'].iloc[0]:.4f}  "
             f"DA={test_metrics['DA'].iloc[0]:.2%}  "
             f"R2={test_metrics['R2'].iloc[0]:.4f}")

    os.makedirs(METRICS_DIR, exist_ok=True)
    metrics_df.to_csv(
        os.path.join(METRICS_DIR, f"ARIMAX_{ticker}_metrics.csv"), index=False
    )

    # oof predictions saved for downstream ridge meta-learner stacking
    os.makedirs(OOF_DIR, exist_ok=True)
    oof_df.to_csv(
        os.path.join(OOF_DIR, f"oof_arimax_{ticker}.csv"), index=False
    )

    os.makedirs(RESULTS_DIR, exist_ok=True)
    test_result_df.to_csv(
        os.path.join(RESULTS_DIR, f"ARIMAX_{ticker}_predictions.csv"), index=False
    )

    plot_results(test_result_df, save_dir=FIGURES_DIR)

    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"arimax_{ticker}.pkl"), "wb") as f:
        pickle.dump({"result": final_result, "order": order, "ticker": ticker}, f)

    log.info(f"  All outputs saved for {ticker}")
    return metrics_df

In [ ]:
all_metrics = []

for ticker in SYMBOLS:
    try:
        m = train_one_ticker(ticker)
        all_metrics.append(m)
    except Exception as e:
        log.error(f"{ticker} FAILED: {e}")

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
    summary_df.to_csv(SUMMARY_METRICS, index=False)
    log.info(f"\nSummary saved: {SUMMARY_METRICS}")
    print("\n" + summary_df.to_string(index=False))

log.info("Done.")